# Descarga Resultados Elecciones

In [5]:
from vothmo.config import elecciones_url
import requests as r
from bs4 import BeautifulSoup


response = r.get(elecciones_url)
response.raise_for_status()

# Filenames
soup = BeautifulSoup(response.content, "lxml")

file_ref = soup.find("select")
files = file_ref.find_all("option")

filename = [f"{f.text}.xlsx" for f in files if f.text != ""]

# Normalize Filenames

def filenorm(files):
    norm_names = []
    for name in files:
        try:
            name.index("Ñ")
            name = name.replace("Ñ","N")
            norm_names.append(name)
        except:
            try:
                name.index("Ó")
                name= name.replace("Ó","O")
                norm_names.append(name)
            except:
                norm_names.append(name)
    return norm_names

elecciones_names = filenorm(filename)

# Elecciones por Distrito 2018, 2021

In [11]:
from vothmo.config import FOLDER_RAW
from yarl import  URL
import os

year = [2018, 2021]

folder = FOLDER_RAW
for y in year:
    for file in elecciones_names:
        download_url = URL(f"https://ieesonora.org.mx/_elecciones/procesos/{y}/resultados/basedatos/")
        name = file.replace(" ", "_") if y==2018 else file
        down_req = download_url / name
        savepath = folder / f"{y}"
        if not os.path.exists(savepath):
            os.mkdir(savepath)
        response = r.get(down_req)
        response.raise_for_status()
        with open(savepath / name, "wb") as distrito:
            distrito.write(response.content)

In [12]:
download_url= URL("https://www.ieesonora.org.mx/elecciones2024/computo/CDE/")
savepath = folder  / "2024"
if not os.path.exists(savepath):
    os.mkdir(savepath)
for name in filename:
    down_req = download_url / name
    response = r.get(down_req)
    response.raise_for_status()
    with open(savepath / name, "wb") as distrito:
        distrito.write(response.content)